# 03 - Local Structure Descriptors and Coordination Chemistry

The next scientific step is to connect energetics with local structure.

Here we use the OnePiece-compatible `DFTDataFrame` tools directly for
structure-derived descriptors:

- `OP.distance_from_surface(...)`
- `OP.get_GCN(...)`

The purpose is not to calculate every imaginable descriptor, but to show how a
materials thesis can move from names and formulas to local atomic environments.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Focus on converged adsorption structures

In [ ]:
analysis = adsorption_energy_conventions(assign_clean_references(df))
ads = analysis[analysis["record_class"].eq("adsorbate")].copy()
ads = OP.converged(ads[ads["E"].notna() & ads["clean_ref_E"].notna()], force_col="fmax", convergence_threshold=0.01)
ads = OP.unify_adsorbates(ads, "HCOOH", "COOH", "adsorbate")
ads = ads[ads["has_structure"]].copy()
ads[["Name", "adsorbate", "material_family", "facet", "E", "fmax"]].head(15)

## 2. Distance of the adsorbate from the surface

In [ ]:
focused = ads[ads["adsorbate"].isin(["CO", "CH3O", "HCOO", "COOH"])].copy()
focused = focused.head(350).copy()  # structural descriptor pass on a representative subset
focused["Distance"] = focused.apply(
    OP.distance_from_surface,
    axis=1,
    struc="struc",
    adsorbate_atoms=["C", "O", "H"],
)
focused[["Name", "adsorbate", "Distance", "facet", "material_family"]].head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for adsorbate, group in focused.groupby("adsorbate"):
    ax.hist(group["Distance"], bins=30, alpha=0.45, label=adsorbate)
ax.set_title("Adsorbate-to-surface distance distribution")
ax.set_xlabel("minimum adsorbate-surface distance / Å")
ax.set_ylabel("count")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Generalized coordination numbers from the local structure

In [ ]:
def gcn_summary(row):
    try:
        gcn = OP.get_GCN(row, cutoff=3.0, structure_column="struc")
    except Exception:
        return pd.Series({"GCN_mean": np.nan, "GCN_min": np.nan, "GCN_max": np.nan})
    if len(gcn) == 0:
        return pd.Series({"GCN_mean": np.nan, "GCN_min": np.nan, "GCN_max": np.nan})
    return pd.Series({
        "GCN_mean": float(np.mean(gcn)),
        "GCN_min": float(np.min(gcn)),
        "GCN_max": float(np.max(gcn)),
    })

gcn_table = focused.apply(gcn_summary, axis=1)
focused = pd.concat([focused, gcn_table], axis=1)
focused[["Name", "adsorbate", "Distance", "GCN_mean", "GCN_min", "GCN_max"]].head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(focused["GCN_mean"].dropna(), bins=30, color="#72b7b2", alpha=0.85)
axes[0].set_title("Distribution of mean GCN")
axes[0].set_xlabel("mean GCN")
axes[0].set_ylabel("count")

co_focus = focused[focused["adsorbate"].eq("CO") & focused["E_ads_CO"].notna()].copy()
scatter = axes[1].scatter(
    co_focus["GCN_mean"],
    co_focus["E_ads_CO"],
    c=co_focus["Distance"],
    cmap="viridis",
    s=45,
)
axes[1].set_title("CO adsorption vs local coordination")
axes[1].set_xlabel("mean GCN")
axes[1].set_ylabel("E_ads(CO) / eV")
fig.colorbar(scatter, ax=axes[1], label="distance / Å")
plt.tight_layout()
plt.show()

## 4. Lowest-energy representatives within each adsorbate family

In [ ]:
descriptor_ready = focused.copy()
descriptor_ready["adsorbate_surface"] = descriptor_ready["adsorbate"] + "::" + descriptor_ready["surface_key"]
descriptor_ready["descriptor_energy"] = descriptor_ready[
    ["E_ads_CO", "E_ads_CH3O_from_CH3OH", "E_ads_HCOO_from_HCOOH", "E_ads_COOH_from_HCOOH"]
].bfill(axis=1).iloc[:, 0]
lowest = OP.group_min(
    descriptor_ready[descriptor_ready["descriptor_energy"].notna()].copy(),
    group="adsorbate_surface",
    value="descriptor_energy",
)
lowest[["Name", "adsorbate", "material_family", "facet", "Distance", "GCN_mean", "descriptor_energy"]].head(30)

This notebook establishes a key materials-science bridge:

energetic trends can now be discussed alongside local coordination and
adsorbate-surface distance. That is exactly the level where mechanistic
arguments in a thesis become stronger than simple ranking tables.